In [2]:
import psycopg2
from match import *
from tools import read_csv_from_gcs

warnings.filterwarnings('ignore')

In [3]:
raw_13 = read_csv_from_gcs(
    bucket_name="ma-data-dev",
    file_name="ADEME/RAW/ADEME_RAW_SUBREGION_CODE_13"
)

raw_75 = read_csv_from_gcs(
    bucket_name="ma-data-dev",
    file_name="ADEME/RAW/ADEME_RAW_SUBREGION_CODE_75"
)

In [5]:
dpe_13 = read_csv_from_gcs(
    bucket_name="ma-data-dev",
    file_name="ADEME/DPE/ADEME_GEOCODED_SUBREGION_CODE_13"
)

dpe_75 = read_csv_from_gcs(
    bucket_name="ma-data-dev",
    file_name="ADEME/DPE/ADEME_GEOCODED_SUBREGION_CODE_75"
)

dpe_13['area'] = dpe_13.surface_habitable_logement.apply(lambda x: round(x, 2))
dpe_75['area'] = dpe_75.surface_habitable_logement.apply(lambda x: round(x, 2))

In [8]:
#Connection BV
host = 'localhost'
user = 'meilleursagents'
db_name = 'meilleursagents_prod'
port = '3333'
password_bv = '85Tg87bz7X53zhYg39hsktU2EegSt7sBdDzwhk7F7nt'
uri_bv = f'postgresql://{user}@{host}:{port}/{db_name}'

conn_bv = psycopg2.connect(uri_bv, password=password_bv)

In [30]:
def decode_geo(
    subregion_cog: int = 75,
    uri: str = conn_bv
):

    sql = f"""
    SELECT
        id,
        cog
    FROM geo.geo_place2
    WHERE
        cog = '{subregion_cog}'
        AND type = 30
        AND enabled = True
        AND active = True
    """

    return pd.read_sql(
        sql,
        uri
    ).id.iloc[0]


In [9]:
def get_observation_bv(
    subregion_id: int = 114,
    date_min: str = '2024-01-01',
    uri: str = conn_bv
):
    sql = """
    SELECT
        past_sales.id,
        past_sales.address_id,
        address.street_id,
        address.subregion_id,
        past_sales.area,
        past_sales.floor,
        past_sales.energy_consumption_classification,
        past_sales.sale_agreement_date,
        past_sales.type
    FROM public.biz_past_sale AS past_sales
    LEFT JOIN geo.geo_address2 address ON (
        past_sales.address_id = address.id
        AND address.enabled = True
    )
    WHERE
        address.subregion_id = %(subregion_id)s
        AND past_sales.sale_agreement_date::DATE >= %(date_min)s
        AND past_sales.sale_agreement_date::DATE < '2025-05-01'
    """

    table = pd.read_sql(
        sql,
        uri,
        params={"subregion_id": subregion_id, "date_min": date_min},
    ).sort_values(by=['sale_agreement_date'], ascending=False)

    table['item_type'] = table['type'].map({1:"APARTMENT", 2:"HOUSE"})

    return table.dropna(subset=['item_type'])

In [10]:
data_paris = get_observation_bv(
    subregion_id=114,
    date_min='2024-01-01',
).dropna(subset=['area'])

In [12]:
master, historic, distribution, statistics = match_test(
    df=data_paris,
    searcher_df=dpe_75,
    saved=None
)

2025-06-11 16:34:15,284 - Processed 1000 rows
2025-06-11 16:34:15,284 - Processed 1000 rows
2025-06-11 16:34:19,766 - Processed 2000 rows
2025-06-11 16:34:19,766 - Processed 2000 rows
2025-06-11 16:34:24,133 - Processed 3000 rows
2025-06-11 16:34:24,133 - Processed 3000 rows
2025-06-11 16:34:29,192 - Processed 4000 rows
2025-06-11 16:34:29,192 - Processed 4000 rows
2025-06-11 16:34:33,516 - Processed 5000 rows
2025-06-11 16:34:33,516 - Processed 5000 rows
2025-06-11 16:34:37,810 - Processed 6000 rows
2025-06-11 16:34:37,810 - Processed 6000 rows
2025-06-11 16:34:42,100 - Processed 7000 rows
2025-06-11 16:34:42,100 - Processed 7000 rows
2025-06-11 16:34:47,271 - Processed 8000 rows
2025-06-11 16:34:47,271 - Processed 8000 rows
2025-06-11 16:34:52,820 - Processed 9000 rows
2025-06-11 16:34:52,820 - Processed 9000 rows
2025-06-11 16:34:57,132 - Processed 10000 rows
2025-06-11 16:34:57,132 - Processed 10000 rows
2025-06-11 16:35:01,443 - Processed 11000 rows
2025-06-11 16:35:01,443 - Proce

📁 Created new file: gs://ma-data-dev/ADEME/AVM_DATA/75/SALES_ENRICHED_SUBREGION_CODE_75.csv
📁 Created new file: gs://ma-data-dev/ADEME/AVM_DATA/75/SALES_ENRICHED_HISTORIC_SUBREGION_CODE_75.csv
📁 Created new file: gs://ma-data-dev/ADEME/AVM_DATA/75/SALES_ENRICHED_DISTRIBUTION_SUBREGION_CODE_75.csv
📁 Created new file: gs://ma-data-dev/ADEME/AVM_DATA/75/SALES_ENRICHED_STATISTICS_SUBREGION_CODE_75.csv


In [13]:
paris_declared = master[[
    'sale_id',
    'address_id',
    'id_aviv',
    'disclosed_dpe',
    'ademe_dpe',
    'etiquette_dpe',
    'etiquette_ges',
    'numero_dpe',
    'match_score',
    'area',
    'ademe_area',
    'floor',
    'ademe_floor',
    'complement_adresse_logement',
    'date_visite_diagnostiqueur',
    'sale_agreement_date',
    'type_key'
]]

In [14]:
paris_declared['error'] = (paris_declared.disclosed_dpe != paris_declared.ademe_dpe).astype(float)
paris_declared['positive_error'] = (paris_declared.disclosed_dpe.str.lower() > paris_declared.ademe_dpe.str.lower()).astype(float)
paris_declared['negative_error'] = (paris_declared.disclosed_dpe.str.lower() < paris_declared.ademe_dpe.str.lower()).astype(float)

In [15]:
positve_error_paris = paris_declared[ (paris_declared.positive_error == 1) & (~paris_declared.disclosed_dpe.isna()) ]
negative_error_paris = paris_declared[ (paris_declared.negative_error == 1) & (~paris_declared.disclosed_dpe.isna()) ]

In [16]:
error_paris = paris_declared[(~paris_declared.ademe_dpe.isna()) & (~paris_declared.disclosed_dpe.isna()) ]

In [17]:
print('--------------------------------------------------------------------------------------------------------------------------------------- ')
print(f'Périmètre: Bien Vendus ayant déclaré un DPE | Mai 2024 - Mai 2025 | {len(data_paris)} ventes')
print('--------------------------------------------------------------------------------------------------------------------------------------- ')

print("% des ventes pour lesquelles on retrouve un DPE dans l'ADEME:", round((1-paris_declared.ademe_dpe.isna().sum()/len(paris_declared))*100 ,1), '%')
print("% de ventes pour lesquelles le DPE déclaré n'est pas celui qu'on retrouve dans l'ADEME:", round(error_paris.error.sum()/len(paris_declared[~paris_declared.disclosed_dpe.isna()])*100,1), '%')
print("% de ventes pour lesquelles le DPE déclaré n'est pas celui qu'on retrouve dans l'ADEME et il est avantageux pour le propriétaire:", round(len(positve_error_paris)/len(paris_declared[~paris_declared.disclosed_dpe.isna()])*100,1), '%')
print("% de ventes pour lesquelles le DPE déclaré n'est pas celui qu'on retrouve dans l'ADEME et il est pénalisant pour le propriétaire:", round(len(negative_error_paris)/len(paris_declared[~paris_declared.disclosed_dpe.isna()])*100,1), '%')

print('--------------------------------------------------------------------------------------------------------------------------------------- ')

--------------------------------------------------------------------------------------------------------------------------------------- 
Périmètre: Bien Vendus ayant déclaré un DPE | Mai 2024 - Mai 2025 | 15980 ventes
--------------------------------------------------------------------------------------------------------------------------------------- 
% des ventes pour lesquelles on retrouve un DPE dans l'ADEME: 81.3 %
% de ventes pour lesquelles le DPE déclaré n'est pas celui qu'on retrouve dans l'ADEME: 18.8 %
% de ventes pour lesquelles le DPE déclaré n'est pas celui qu'on retrouve dans l'ADEME et il est avantageux pour le propriétaire: 10.6 %
% de ventes pour lesquelles le DPE déclaré n'est pas celui qu'on retrouve dans l'ADEME et il est pénalisant pour le propriétaire: 8.2 %
--------------------------------------------------------------------------------------------------------------------------------------- 
